# 🍎 AlphaApple Training V4 (Expert Iteration)

**V4 핵심 개선사항**: Self-Improvement Loop (Expert Iteration)

## 🎯 V3 문제점 분석
- **V3 결과**: 평균 101.9개 (59.9%)
- **Expert 보상**: 평균 103.7개
- **문제**: Expert가 약함 → 학습 ceiling
- Multiple Rollouts로 데이터 다양성 증가 시도했지만, Expert 자체가 약해서 한계

## 💡 V4 해결책: Expert Iteration

**핵심 아이디어**: 정책이 스스로를 가르침!

```
반복 (3-5회):
  1. 현재 정책으로 많은 보드 플레이 (각 보드당 5 rollouts)
  2. 고품질 에피소드만 선택 (top 50% 또는 threshold 이상)
  3. 선택된 데이터로 정책 학습
  4. 평가 → 개선 확인
```

**왜 작동하는가?**
- 초기: 약한 정책 (100개)
- Iteration 1: 운 좋게 105개 달성 → 이것만 학습 → 정책 개선
- Iteration 2: 개선된 정책으로 110개 달성 → 학습 → 더 개선
- Iteration N: 점진적으로 성능 향상 (self-play)

**AlphaGo Zero의 핵심 메커니즘**

## 🎯 최종 목표
**170점 만점 (100% 클리어)** 🍎

사람 최고(130개)는 단순 참고일 뿐!

## 🔧 Setup

In [ ]:
# Colab 환경 확인
try:
    import google.colab
    IN_COLAB = True
    print("✅ Running in Colab")
except:
    IN_COLAB = False
    print("❌ Not in Colab")

# GPU 확인
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# GitHub에서 코드 가져오기
if IN_COLAB:
    !git clone https://github.com/kbsooo/AlphaApple.git
    %cd AlphaApple
    !git checkout claude/alphaapple-v3-training-011CUzSDzYASVTysjH6djd4o
else:
    import os
    os.chdir('/home/user/AlphaApple')

In [ ]:
# 의존성 설치
!pip install -q gymnasium numpy torch tqdm

## 🎯 1. Expert Iteration 핵심 함수

**Self-Play로 데이터 수집 + 고품질만 선택**

In [ ]:
import sys
import numpy as np
import pickle
from tqdm.notebook import tqdm

sys.path.insert(0, '.')

from envs.fruitbox_env import FruitBoxEnv, FruitBoxConfig
from envs.backward_generator import BackwardBoardGenerator
from envs.autoregressive_wrapper import make_autoregressive_env

In [ ]:
def play_with_policy(
    policy,
    initial_board,
    temperature=1.0,
    device='cuda'
):
    """
    정책으로 플레이 (temperature sampling)
    
    Args:
        policy: 학습된 정책
        initial_board: 초기 보드
        temperature: 높을수록 탐색적
        device: cuda/cpu
    
    Returns:
        history: [(obs, action, reward, mask), ...]
        total_reward: 최종 보상
    """
    wrapped_env = make_autoregressive_env(rows=10, cols=17)
    env = wrapped_env.env
    env.board = initial_board.copy().astype(np.int16)
    obs = initial_board.clip(0, 9).astype(np.int8)
    
    history = []
    total_reward = 0
    steps = 0
    
    policy.eval()
    with torch.no_grad():
        while steps < 500:
            # 합법 행동 확인
            legal = env.legal_actions()
            if len(legal) == 0:
                break
            
            # Masks
            masks_np = wrapped_env.get_autoregressive_masks()
            masks_torch = {
                'r1_mask': torch.from_numpy(masks_np['r1_mask']).to(device),
                'c1_masks': torch.from_numpy(masks_np['c1_masks']).to(device),
                'r2_masks': torch.from_numpy(masks_np['r2_masks']).to(device),
                'c2_masks': torch.from_numpy(masks_np['c2_masks']).to(device)
            }
            
            # 정책으로 행동 샘플링
            obs_tensor = torch.from_numpy(obs).float().unsqueeze(0).unsqueeze(0).to(device)
            
            # Temperature 적용
            if temperature == 0:
                action_tuple, _, _, _ = policy(obs_tensor, deterministic=True, masks=masks_torch)
            else:
                # Temperature sampling
                action_tuple, _, _, _ = policy(obs_tensor, deterministic=False, masks=masks_torch, temperature=temperature)
            
            r1 = int(action_tuple[0][0].item())
            c1 = int(action_tuple[1][0].item())
            r2 = int(action_tuple[2][0].item())
            c2 = int(action_tuple[3][0].item())
            
            # 기록
            history.append((obs.copy(), (r1, c1, r2, c2), 0, masks_np))
            
            # Step
            obs, reward, terminated, truncated, info = wrapped_env.step_with_coords(r1, c1, r2, c2)
            
            history[-1] = (history[-1][0], history[-1][1], reward, history[-1][3])
            total_reward += reward
            steps += 1
            
            if terminated or truncated:
                break
    
    return history, total_reward


def collect_self_play_data(
    policy,
    n_episodes=500,
    n_rollouts=5,
    target_coverage=0.95,
    device='cuda',
    quality_threshold=None,
    top_k_pct=0.5
):
    """
    현재 정책으로 Self-Play 데이터 수집
    
    Args:
        policy: 현재 정책
        n_episodes: 보드 수
        n_rollouts: 각 보드당 플레이 횟수
        quality_threshold: 최소 보상 (None이면 top_k_pct 사용)
        top_k_pct: 상위 몇 %만 선택 (기본 50%)
    
    Returns:
        episodes: 고품질 에피소드만
    """
    all_episodes = []
    all_rewards = []
    
    # 다양한 temperature
    temperatures = [0.5, 0.7, 1.0, 1.3, 1.5]
    
    for i in tqdm(range(n_episodes), desc="Self-Play Data Collection"):
        # 역방향 생성
        generator = BackwardBoardGenerator(rows=10, cols=17, seed=i)
        board, solution = generator.generate(target_coverage=target_coverage)
        
        # 여러 번 플레이
        best_history = None
        best_reward = -float('inf')
        
        for t in range(n_rollouts):
            temp = temperatures[t % len(temperatures)]
            history, reward = play_with_policy(policy, board, temperature=temp, device=device)
            
            if reward > best_reward:
                best_reward = reward
                best_history = history
        
        if best_history is None or len(best_history) == 0:
            continue
        
        # 데이터 저장
        observations = []
        actions = []
        rewards = []
        masks = []
        
        for obs, action, reward, mask in best_history:
            observations.append(obs)
            actions.append(action)
            rewards.append(reward)
            masks.append(mask)
        
        all_episodes.append({
            'observations': np.array(observations),
            'actions': np.array(actions),
            'rewards': np.array(rewards),
            'masks': masks,
            'total_reward': best_reward,
            'steps': len(best_history),
            'seed': i,
        })
        all_rewards.append(best_reward)
    
    # 고품질만 선택
    if quality_threshold is not None:
        # Threshold 기반
        selected_episodes = [ep for ep in all_episodes if ep['total_reward'] >= quality_threshold]
        print(f"\n=== Quality Threshold: {quality_threshold} ===")
    else:
        # Top K% 기반
        sorted_episodes = sorted(all_episodes, key=lambda x: x['total_reward'], reverse=True)
        k = int(len(sorted_episodes) * top_k_pct)
        selected_episodes = sorted_episodes[:k]
        threshold = selected_episodes[-1]['total_reward'] if selected_episodes else 0
        print(f"\n=== Top {top_k_pct*100:.0f}% Selection (threshold: {threshold:.0f}) ===")
    
    selected_rewards = [ep['total_reward'] for ep in selected_episodes]
    
    print(f"전체 에피소드: {len(all_episodes)}")
    print(f"선택된 에피소드: {len(selected_episodes)} ({len(selected_episodes)/len(all_episodes)*100:.1f}%)")
    print(f"전체 평균 보상: {np.mean(all_rewards):.1f} ± {np.std(all_rewards):.1f}")
    print(f"선택된 평균 보상: {np.mean(selected_rewards):.1f} ± {np.std(selected_rewards):.1f}")
    print(f"최대 보상: {max(all_rewards):.0f}")
    print(f"총 transition: {sum(ep['steps'] for ep in selected_episodes)}")
    
    return selected_episodes

## 🧠 2. 모델 초기화

**옵션 1**: 처음부터 학습  
**옵션 2**: V3 모델을 warm-start로 사용 (있다면)

In [ ]:
from models.lightweight_policy import LightweightPolicy

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 모델 생성
policy = LightweightPolicy(rows=10, cols=17, latent_dim=128)

# V3 모델이 있으면 로드 (선택사항)
try:
    policy.load_state_dict(torch.load('bc_policy_best_v3.pt'))
    print("✅ V3 모델 로드 성공 (warm-start)")
except:
    print("⚠️  V3 모델 없음 → 처음부터 학습")

policy = policy.to(device)
print(f"파라미터 수: {sum(p.numel() for p in policy.parameters()):,}")
print(f"Device: {device}")

## 🔄 3. Expert Iteration Loop

**Self-Improvement의 핵심!**

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

class ExpertDatasetWithMasks(Dataset):
    def __init__(self, episodes):
        self.data = []
        
        for ep in episodes:
            for t in range(len(ep['observations'])):
                self.data.append({
                    'obs': ep['observations'][t],
                    'action': ep['actions'][t],
                    'mask': ep['masks'][t]
                })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        obs = torch.from_numpy(item['obs']).float().unsqueeze(0)
        act = torch.from_numpy(np.array(item['action'])).long()
        
        masks = item['mask']
        masks_torch = {
            'r1_mask': torch.from_numpy(masks['r1_mask']),
            'c1_masks': torch.from_numpy(masks['c1_masks']),
            'r2_masks': torch.from_numpy(masks['r2_masks']),
            'c2_masks': torch.from_numpy(masks['c2_masks'])
        }
        
        return obs, act, masks_torch


def train_on_data(policy, episodes, n_epochs=10, batch_size=128, lr=3e-4, device='cuda'):
    """
    주어진 데이터로 정책 학습
    """
    dataset = ExpertDatasetWithMasks(episodes)
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    
    train_dataset, val_dataset = torch.utils.data.random_split(
        dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    best_val_loss = float('inf')
    
    for epoch in range(n_epochs):
        # Train
        policy.train()
        train_loss = 0
        train_batches = 0
        
        for batch_obs, batch_act, batch_masks in train_loader:
            batch_obs = batch_obs.to(device)
            batch_act = batch_act.to(device)
            
            batch_masks_device = {
                'r1_mask': batch_masks['r1_mask'].to(device),
                'c1_masks': batch_masks['c1_masks'].to(device),
                'r2_masks': batch_masks['r2_masks'].to(device),
                'c2_masks': batch_masks['c2_masks'].to(device)
            }
            
            action_tuple = tuple(batch_act[:, i] for i in range(4))
            _, log_prob, _, _ = policy(batch_obs, action=action_tuple, masks=batch_masks_device)
            
            loss = -log_prob.mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_batches += 1
        
        train_loss /= train_batches
        
        # Val
        policy.eval()
        val_loss = 0
        val_batches = 0
        
        with torch.no_grad():
            for batch_obs, batch_act, batch_masks in val_loader:
                batch_obs = batch_obs.to(device)
                batch_act = batch_act.to(device)
                
                batch_masks_device = {
                    'r1_mask': batch_masks['r1_mask'].to(device),
                    'c1_masks': batch_masks['c1_masks'].to(device),
                    'r2_masks': batch_masks['r2_masks'].to(device),
                    'c2_masks': batch_masks['c2_masks'].to(device)
                }
                
                action_tuple = tuple(batch_act[:, i] for i in range(4))
                _, log_prob, _, _ = policy(batch_obs, action=action_tuple, masks=batch_masks_device)
                
                loss = -log_prob.mean()
                val_loss += loss.item()
                val_batches += 1
        
        val_loss /= val_batches
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
        
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{n_epochs}: Train={train_loss:.4f}, Val={val_loss:.4f}")
    
    return best_val_loss

In [ ]:
def evaluate_policy_quick(policy, n_episodes=20, target_coverage=0.95, device='cuda'):
    """
    빠른 평가 (20 episodes)
    """
    episode_rewards = []
    
    with torch.no_grad():
        for i in range(n_episodes):
            generator = BackwardBoardGenerator(rows=10, cols=17, seed=10000+i)
            board, _ = generator.generate(target_coverage=target_coverage)
            
            wrapped_env = make_autoregressive_env(rows=10, cols=17)
            env = wrapped_env.env
            env.board = board.astype(np.int16)
            obs = board.clip(0, 9).astype(np.int8)
            
            episode_reward = 0
            steps = 0
            
            while True:
                masks_np = wrapped_env.get_autoregressive_masks()
                masks_torch = {
                    'r1_mask': torch.from_numpy(masks_np['r1_mask']).to(device),
                    'c1_masks': torch.from_numpy(masks_np['c1_masks']).to(device),
                    'r2_masks': torch.from_numpy(masks_np['r2_masks']).to(device),
                    'c2_masks': torch.from_numpy(masks_np['c2_masks']).to(device)
                }
                
                obs_tensor = torch.from_numpy(obs).float().unsqueeze(0).unsqueeze(0).to(device)
                action_tuple, _, _, _ = policy(obs_tensor, deterministic=True, masks=masks_torch)
                
                r1 = int(action_tuple[0][0].item())
                c1 = int(action_tuple[1][0].item())
                r2 = int(action_tuple[2][0].item())
                c2 = int(action_tuple[3][0].item())
                
                obs, reward, terminated, truncated, info = wrapped_env.step_with_coords(r1, c1, r2, c2)
                
                episode_reward += reward
                steps += 1
                
                if terminated or truncated or steps >= 500:
                    break
            
            episode_rewards.append(episode_reward)
    
    return episode_rewards

### Expert Iteration 메인 루프

In [ ]:
# Expert Iteration 설정
N_ITERATIONS = 5
N_EPISODES_PER_ITER = 500
N_ROLLOUTS = 5
TOP_K_PCT = 0.5  # 상위 50%만 선택
N_EPOCHS_PER_ITER = 15

iteration_results = []

print("="*70)
print("🍎 AlphaApple V4: Expert Iteration")
print("="*70)
print(f"Iterations: {N_ITERATIONS}")
print(f"Episodes per iteration: {N_EPISODES_PER_ITER}")
print(f"Rollouts per episode: {N_ROLLOUTS}")
print(f"Selection: Top {TOP_K_PCT*100:.0f}%")
print(f"Training epochs: {N_EPOCHS_PER_ITER}")
print("="*70)
print()

for iteration in range(N_ITERATIONS):
    print(f"\n{'='*70}")
    print(f"📍 Iteration {iteration + 1}/{N_ITERATIONS}")
    print(f"{'='*70}")
    
    # 1. Self-Play 데이터 수집
    print(f"\n[Step 1] Self-Play Data Collection...")
    episodes = collect_self_play_data(
        policy=policy,
        n_episodes=N_EPISODES_PER_ITER,
        n_rollouts=N_ROLLOUTS,
        target_coverage=0.95,
        device=device,
        top_k_pct=TOP_K_PCT
    )
    
    if len(episodes) == 0:
        print("⚠️  선택된 에피소드 없음. 반복 중단.")
        break
    
    # 2. 정책 학습
    print(f"\n[Step 2] Training Policy...")
    best_val_loss = train_on_data(
        policy=policy,
        episodes=episodes,
        n_epochs=N_EPOCHS_PER_ITER,
        batch_size=128,
        lr=3e-4,
        device=device
    )
    
    # 3. 평가
    print(f"\n[Step 3] Evaluating...")
    eval_rewards = evaluate_policy_quick(policy, n_episodes=30, device=device)
    
    avg_reward = np.mean(eval_rewards)
    max_reward = max(eval_rewards)
    
    print(f"\n✅ Iteration {iteration + 1} 완료:")
    print(f"   평균 보상: {avg_reward:.1f}개 ({avg_reward/170*100:.1f}%)")
    print(f"   최대 보상: {max_reward:.0f}개 ({max_reward/170*100:.1f}%)")
    print(f"   Val Loss: {best_val_loss:.4f}")
    
    # 결과 저장
    iteration_results.append({
        'iteration': iteration + 1,
        'avg_reward': avg_reward,
        'max_reward': max_reward,
        'val_loss': best_val_loss,
        'n_episodes': len(episodes)
    })
    
    # 모델 저장
    torch.save(policy.state_dict(), f'bc_policy_v4_iter{iteration+1}.pt')
    print(f"   모델 저장: bc_policy_v4_iter{iteration+1}.pt")

print("\n" + "="*70)
print("🎉 Expert Iteration 완료!")
print("="*70)

## 📊 4. 최종 평가 (50 episodes)

In [ ]:
# 최고 성능 모델 선택
best_iter = max(iteration_results, key=lambda x: x['avg_reward'])
print(f"최고 성능 iteration: {best_iter['iteration']}")

# 로드
policy.load_state_dict(torch.load(f"bc_policy_v4_iter{best_iter['iteration']}.pt"))
policy.eval()

# 평가
print("\n=== 95% 제거 가능 보드 평가 (50 episodes) ===")
results_95 = evaluate_policy_quick(policy, n_episodes=50, target_coverage=0.95, device=device)

print(f"평균: {np.mean(results_95):.1f} ± {np.std(results_95):.1f}")
print(f"최대: {max(results_95):.0f}/170 ({max(results_95)/170*100:.1f}%)")
print(f"최소: {min(results_95):.0f}")
print(f"범위: [{min(results_95):.0f}, {max(results_95):.0f}]")

# 최종 모델 저장
torch.save(policy.state_dict(), 'bc_policy_best_v4.pt')
print("\n✅ 최종 모델 저장: bc_policy_best_v4.pt")

## 📊 5. 결과 요약 (Claude Code용)

In [ ]:
print("\n" + "="*70)
print("🍎 AlphaApple V4 Training Summary (Claude Code용)")
print("="*70)
print()
print("[1] Expert Iteration 설정")
print(f"  - Iterations: {N_ITERATIONS}")
print(f"  - Episodes per iteration: {N_EPISODES_PER_ITER}")
print(f"  - Selection: Top {TOP_K_PCT*100:.0f}%")
print()
print("[2] Iteration별 성능 추이")
print("  | Iter | 평균 | 최대 | Val Loss |")
print("  |------|------|------|----------|")
for result in iteration_results:
    print(f"  | {result['iteration']:4d} | {result['avg_reward']:4.1f} | {result['max_reward']:4.0f} | {result['val_loss']:.4f} |")
print()
print("[3] 최종 평가 결과 (50 episodes)")
print(f"  평균: {np.mean(results_95):.1f}개 ({np.mean(results_95)/170*100:.1f}%)")
print(f"  최대: {max(results_95):.0f}개 ({max(results_95)/170*100:.1f}%)")
print(f"  범위: [{min(results_95):.0f}, {max(results_95):.0f}]")
print()
print("[4] 버전 비교")
print("  | 버전 | 방법 | 평균 (95%) | 최대 |")
print("  |------|------|-----------|------|")
print("  | V1   | BC (no mask) | -500 (0%) | 0    |")
print("  | V2   | BC + Mask | 101.5 (59.7%) | 129  |")
print("  | V3   | Multi-Rollout | 101.9 (59.9%) | 120  |")
print(f"  | V4   | Expert Iteration | {np.mean(results_95):.1f} ({np.mean(results_95)/170*100:.1f}%) | {max(results_95):.0f}  |")
print()
print("[5] 목표 대비 (최종 목표: 170점 만점)")
print(f"  V4 최고:   {max(results_95):.0f}개 ({max(results_95)/170*100:.1f}%)")
print(f"  목표까지:  {170 - max(results_95):.0f}개 남음")
print(f"  진척률:    {max(results_95)/170*100:.1f}%")
print()
print("[6] Self-Improvement 효과")
if len(iteration_results) > 1:
    initial_avg = iteration_results[0]['avg_reward']
    final_avg = iteration_results[-1]['avg_reward']
    improvement = final_avg - initial_avg
    print(f"  초기 평균: {initial_avg:.1f}개")
    print(f"  최종 평균: {final_avg:.1f}개")
    print(f"  개선량:    +{improvement:.1f}개 ({improvement/initial_avg*100:.1f}%)")
else:
    print("  (단일 iteration)")
print()
print("[7] 다음 단계 제안")
if max(results_95) >= 140:
    print("  🎯 거의 도달! MCTS로 완벽 플레이 시도")
elif max(results_95) >= 115:
    print("  📈 좋은 진전! PPO Fine-tuning으로 추가 개선")
else:
    print("  🔄 더 많은 iterations 또는 더 큰 데이터셋 필요")
print()
print("="*70)

## 📈 6. 시각화 (선택사항)

In [ ]:
import matplotlib.pyplot as plt

if len(iteration_results) > 1:
    iters = [r['iteration'] for r in iteration_results]
    avgs = [r['avg_reward'] for r in iteration_results]
    maxs = [r['max_reward'] for r in iteration_results]
    
    plt.figure(figsize=(10, 6))
    plt.plot(iters, avgs, marker='o', label='Average Reward', linewidth=2)
    plt.plot(iters, maxs, marker='s', label='Max Reward', linewidth=2)
    plt.axhline(y=170, color='r', linestyle='--', label='Perfect Play (170)')
    plt.axhline(y=130, color='g', linestyle='--', label='Human Best (130)')
    plt.xlabel('Iteration', fontsize=12)
    plt.ylabel('Reward (cells removed)', fontsize=12)
    plt.title('Expert Iteration: Self-Improvement', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('v4_expert_iteration.png', dpi=150)
    plt.show()
    
    print("✅ 그래프 저장: v4_expert_iteration.png")

## 💾 7. 모델 다운로드 (Colab)

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    # 최종 모델 다운로드
    files.download('bc_policy_best_v4.pt')
    
    # 모든 iteration 모델 다운로드 (선택)
    # for i in range(1, N_ITERATIONS + 1):
    #     files.download(f'bc_policy_v4_iter{i}.pt')
    
    print("✅ 모델 다운로드 완료")